In [66]:
import os

from dotenv import load_dotenv
from langchain_core.messages import trim_messages
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_openai import ChatOpenAI
from pydantic import SecretStr

# 1. 定义一个按 Token 裁剪的函数
def trim_by_tokens(history: ChatMessageHistory, max_tokens=200):
    """将历史消息裁剪到指定的最大 token 数"""
    trimmed = trim_messages(
        history.messages,
        # 关键：这里使用模型的 tokenizer 进行精确计数
        token_counter=model.get_num_tokens_from_messages,
        max_tokens=max_tokens,
        strategy="last",          # 保留最新的消息
        start_on="human",         # 确保从用户消息开始
        include_system=True,      # 始终保留系统提示词
    )
    # 原地更新历史记录
    history.messages[:] = trimmed

# 2. 创建历史记录存储和获取函数
session_store = {}

def get_session_history(session_id):
    if session_id not in session_store:
        session_store[session_id] = ChatMessageHistory()
    
    history = session_store[session_id]
    # 每次获取时，自动将历史裁剪到 200 tokens
    trim_by_tokens(history, max_tokens=100)
    return history
def add_history_message(session_id, message):
    history = get_session_history(session_id)
    history.add_message(message)
# 3. 构建 LCEL 链并包装
load_dotenv(override=True)

model_name = os.getenv("MODEL")
model_base_url = os.getenv("BASE_URL")
model_api_key = os.getenv("API_KEY")
api_key_value = SecretStr(model_api_key) if model_api_key else None

model = ChatOpenAI(
    model=model_name or "gpt-4o-mini",
    base_url=model_base_url or None,
    api_key=api_key_value,
    temperature=0,
 )
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
])

# chain = prompt | model

# # 4. 用 RunnableWithMessageHistory 包装链
# chain_with_history = RunnableWithMessageHistory(
#     runnable=chain,
#     get_session_history=get_session_history,
#     input_messages_key="input",
#     history_messages_key="chat_history",
# )

# prompt = prompt.invoke({"input": "Hello, how are you?", "chat_history": get_session_history("session1").messages})
# print(prompt)


In [67]:
add_history_message("session1", {"role": "human", "content": "Hello, how are you?"})
add_history_message("session1", {"role": "assistant", "content": "I'm good, thank you!"})
add_history_message("session1", {"role": "human", "content": "What's the weather like today?"})
add_history_message("session1", {"role": "assistant", "content": "It's sunny and warm."})
add_history_message("session1", {"role": "human", "content": "What's the weather like today?"})
add_history_message("session1", {"role": "assistant", "content": "It's sunny and warm."})

prompt1 = prompt.invoke({"input": "Can you summarize our conversation?", "chat_history": get_session_history("session1").messages})
# print(prompt)
print(*(m for m in prompt1.messages), sep="\n")

content='You are a helpful assistant.' additional_kwargs={} response_metadata={}
content='Hello, how are you?' additional_kwargs={} response_metadata={}
content="I'm good, thank you!" additional_kwargs={} response_metadata={} tool_calls=[] invalid_tool_calls=[]
content="What's the weather like today?" additional_kwargs={} response_metadata={}
content="It's sunny and warm." additional_kwargs={} response_metadata={} tool_calls=[] invalid_tool_calls=[]
content="What's the weather like today?" additional_kwargs={} response_metadata={}
content="It's sunny and warm." additional_kwargs={} response_metadata={} tool_calls=[] invalid_tool_calls=[]
content='Can you summarize our conversation?' additional_kwargs={} response_metadata={}


In [68]:
print(get_session_history("session1").messages)
chain = prompt | model
chain_with_history = RunnableWithMessageHistory(
    runnable=chain,
    get_session_history=get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history"
)
res = chain_with_history.invoke({"input": "Can you summarize our conversation?"},config={"configurable": {"session_id": "session1"}})

print(type(res))
print(res)
# 计算总的 token 使用量
estimated_tokens = model.get_num_tokens_from_messages(get_session_history("session1").messages)
print(estimated_tokens)  # 例如 23

[HumanMessage(content='Hello, how are you?', additional_kwargs={}, response_metadata={}), AIMessage(content="I'm good, thank you!", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content="What's the weather like today?", additional_kwargs={}, response_metadata={}), AIMessage(content="It's sunny and warm.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content="What's the weather like today?", additional_kwargs={}, response_metadata={}), AIMessage(content="It's sunny and warm.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]
<class 'langchain_core.messages.ai.AIMessage'>
content='You greeted me and asked how I was; I said I was good. Then you asked twice about the weather, and I replied both times that it was sunny and warm.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 50, 'prompt_tokens': 86, 'total_tokens

In [69]:
print(get_session_history("session1").messages)
print(res                    # 取最后一个 AIMessage
.response_metadata           # 元数据
["token_usage"]              # token 信息
["total_tokens"])             # 总 token)

[HumanMessage(content="What's the weather like today?", additional_kwargs={}, response_metadata={}), AIMessage(content="It's sunny and warm.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content="What's the weather like today?", additional_kwargs={}, response_metadata={}), AIMessage(content="It's sunny and warm.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='Can you summarize our conversation?', additional_kwargs={}, response_metadata={}), AIMessage(content='You greeted me and asked how I was; I said I was good. Then you asked twice about the weather, and I replied both times that it was sunny and warm.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 50, 'prompt_tokens': 86, 'total_tokens': 136, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 9, 'rejected_prediction_toke

In [70]:
res = chain_with_history.invoke({"input": "1+1=?"},config={"configurable": {"session_id": "session1"}})
print(res)

content='2' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 113, 'total_tokens': 128, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 8, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4', 'system_fingerprint': None, 'id': 'resp_07235dffd7ec9f5c0169c640e37904819083fd600dc303efaa', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019d2e6d-6ed8-7010-83a8-9b01999866e3-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 113, 'output_tokens': 15, 'total_tokens': 128, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 8}}


In [71]:
print(get_session_history("session1").messages)
a = get_session_history("session1").messages
print(*(f"- {m.content}" for m in a), sep="\n")

[HumanMessage(content="What's the weather like today?", additional_kwargs={}, response_metadata={}), AIMessage(content="It's sunny and warm.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='Can you summarize our conversation?', additional_kwargs={}, response_metadata={}), AIMessage(content='You greeted me and asked how I was; I said I was good. Then you asked twice about the weather, and I replied both times that it was sunny and warm.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 50, 'prompt_tokens': 86, 'total_tokens': 136, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 9, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4', 'system_fingerprint': None, 'id': 'resp_08cb1deef1f4a2ed0169c640dfea0c819a8e2a0ce63a83f334

In [72]:
# 计算总的 token 使用量
estimated_tokens = model.get_num_tokens_from_messages(a)
print(estimated_tokens)  # 例如 23

84


In [73]:
res = chain_with_history.invoke({"input": "讲解一下langchain"},config={"configurable": {"session_id": "session1"}})
print(res)
a = get_session_history("session1").messages
print(*(f"- {m.content}" for m in a), sep="\n")

content='当然可以。LangChain 是一个用于开发基于大语言模型（LLM）应用的框架，核心目标是：\n\n**把“调用模型”这件事，扩展成“构建完整 AI 应用”**。\n\n你可以把它理解为一个“胶水框架”，用来把下面这些东西串起来：\n\n- 大语言模型（如 OpenAI、Anthropic、本地模型）\n- Prompt 模板\n- 记忆模块\n- 外部工具\n- 数据库 / 向量数据库\n- 文档检索\n- Agent 智能体工作流\n\n---\n\n## 一、LangChain 是干什么的？\n\n如果你只是简单问模型一个问题，直接调用 API 就够了，比如：\n\n- 发一个 prompt\n- 得到一个回答\n\n但如果你要做更复杂的应用，比如：\n\n- **问答机器人**：能读取公司知识库回答问题\n- **文档分析系统**：上传 PDF 后自动总结\n- **智能客服**：结合历史对话和业务规则回复\n- **Agent**：让模型自己选择工具、查资料、执行任务\n\n这时候就会涉及很多步骤，LangChain 就是为了让这些步骤更容易组织和复用。\n\n---\n\n## 二、LangChain 的核心概念\n\n### 1. Models（模型）\nLangChain 可以接入各种模型，包括：\n\n- OpenAI\n- Anthropic\n- Google\n- Hugging Face\n- 本地模型（如 Ollama、Llama 等）\n\n它帮你统一了调用方式，减少你频繁改接口的成本。\n\n---\n\n### 2. Prompts（提示模板）\n提示词往往不是写死的，而是动态拼接的。\n\n比如：\n\n- 用户问题\n- 系统角色设定\n- 上下文信息\n- 检索出来的知识\n\nLangChain 提供 PromptTemplate，方便你做变量化提示。\n\n例如：\n\n```python\nfrom langchain.prompts import PromptTemplate\n\nprompt = PromptTemplate.from_template("请用简洁中文回答：{question}")\nprint(prompt.format(question="什么是机器学习？"))\n```\n\

In [74]:
a = get_session_history("session1").messages
print(*(f"- {m.content}" for m in a), sep="\n")